In [15]:
import pandas as pd
import requests
import json
from dotenv import load_dotenv
import os
import pandas as pd
import time
from time import sleep
from itertools import islice
from joblib import Parallel, delayed
from tqdm import tqdm
load_dotenv()
API_KEY = os.getenv("API_KEY")

### Filtering Authors by work count

In [16]:
df1 = pd.read_csv("authors_data.csv")

indexes = df1.index[
    (df1["works_count"] >= 5) &
    (df1["works_count"] <= 5000)
].tolist()

filtered_author_list = df1.loc[indexes, "author_id"]

### Calling API for all works of all authors from Week 2

In [18]:
import pandas as pd
import requests
from itertools import islice
from time import sleep
from joblib import Parallel, delayed
from tqdm import tqdm

# Your existing filters
css_concepts = ["C157449823", "C77805123", "C162324750", "C17744445"]
quant_concepts = ["C33923547", "C121332964", "C41008148"]

concept_filter1 = f"concept.id:{'|'.join(css_concepts)}"
concept_filter2 = f"concept.id:{'|'.join(quant_concepts)}"

BASE_URL = "https://api.openalex.org/works"

def chunked(iterable, size):
    it = iter(iterable)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            break
        yield chunk

#Created function to call it parallel for quicker extraction
def process_single_batch(author_batch, batch_num):
    """Process one batch of authors - this will run in parallel"""
    
    rows2_local = []
    rows3_local = []
    
    author_filter = "author.id:" + "|".join(author_batch)

    filter_string = (
        f"{author_filter},"
        f"cited_by_count:>10,"
        f"authors_count:<10,"
        f"{concept_filter1},"
        f"{concept_filter2}"
    )
    
    cursor = "*"
    
    while cursor:
        params = {
            "filter": filter_string,
            "select": "id,publication_year,cited_by_count,title,abstract_inverted_index,authorships",
            "per-page": 200,
            "cursor": cursor,
            "api_key": API_KEY
        }
        
        try:
            response = requests.get(BASE_URL, params=params)
            response.raise_for_status()
            data = response.json()
        except Exception as e:
            print(f"Error in batch {batch_num}: {e}")
            break
        
        results = data.get("results", [])
        if not results:
            break
        
        for item in results:
            work_id = item.get("id")
            
    
            
            # Extract author IDs
            author_ids = [
                auth["author"]["id"]
                for auth in item.get("authorships", [])
                if auth.get("author") and auth["author"].get("id")
            ]
        
            
            rows2_local.append({
                "id": work_id,
                "publication_year": item.get("publication_year"),
                "cited_by_count": item.get("cited_by_count"),
                "author_ids": author_ids,
            })
            
            rows3_local.append({
                "id": work_id,
                "title": item.get("title"),
                "abstract_index": item.get("abstract_inverted_index"),
            })
        
        cursor = data["meta"].get("next_cursor")
        sleep(0.05)  # Small delay between pages to avoid going over the rate limit 
    
    return rows2_local, rows3_local

# Create batches
author_batches = list(chunked(filtered_author_list, 25))

# Process batches in parallel with tqdm progress bar
results = Parallel(n_jobs=4, verbose=0)(
    delayed(process_single_batch)(batch, i+1) 
    for i, batch in enumerate(tqdm(author_batches, desc="Processing batches"))
)

# Combine results
rows2 = []
rows3 = []
for batch_rows2, batch_rows3 in results:
    rows2.extend(batch_rows2)
    rows3.extend(batch_rows3)

# Create DataFrames
df2 = pd.DataFrame(rows2)
df3 = pd.DataFrame(rows3)

print(f"\nComplete!")
print(f"Works retrieved: {len(df2)}")
print(f"Abstracts retrieved: {len(df3)}")

Processing batches: 100%|██████████| 54/54 [00:24<00:00,  2.19it/s]



Complete!
Works retrieved: 12826
Abstracts retrieved: 12826


In [ ]:
df2.to_csv("IC2S2_papers.csv", index=False)
df3.to_csv("IC2S2_abstracts.csv", index=False)